In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

from delta.tables import DeltaTable


In [0]:
df = spark.read.format("csv")\
       .option("inferSchema",True)\
       .option("header", True)\
       .load("/Volumes/pp_learning/default/data_csv/employee4/")

In [0]:
%sql
select * from pp_learning.silver.employee
where employee_id = 2251


In [0]:
display(df)

In [0]:
df = df.withColumn("processDate",current_timestamp())

In [0]:
display(df)

In [0]:
display(df.filter(col("status").like("%" + "active" + "%")))

In [0]:
display(df.filter(col("status").like("Active")))

In [0]:
%sql
select count(*) from pp_learning.silver.employee

In [0]:
display( df.filter(col("City").like("Delhi")) )

In [0]:
if spark.catalog.tableExists("pp_learning.silver.employee"): \
    dlt_obj = DeltaTable.forName(spark, "pp_learning.silver.employee") \
    dlt_obj.alias("trg").merge(df.alias("src"), "trg.employee_id == src.employee_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
else:
    df.write.format("delta") \
        .mode("append") \
        .saveAsTable("pp_learning.silver.employee")
    


In [0]:
from delta.tables import DeltaTable

table_name = "pp_learning.silver.employee4"

if spark.catalog.tableExists(table_name):

    dlt_obj = DeltaTable.forName(spark, table_name)

    (
        dlt_obj.alias("trg")
        .merge(
            df.alias("src"),
            "trg.employee_id = src.employee_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

In [0]:
%sql
select * from pp_learning.silver.employee

In [0]:
%sql
select employee_id,first_name from pp_learning.silver.employee

In [0]:
%sql
select distinct department from pp_learning.silver.employee

In [0]:
%sql
select * from pp_learning.silver.employee
where salary > 100000;

In [0]:
%sql
select * from pp_learning.silver.employee
where department = 'IT'
and salary > 200000;

In [0]:
%sql
select * from pp_learning.silver.employee
where city in ('Mumbai', 'Bangalore')
and salary > 60000

In [0]:
%sql
select * from pp_learning.silver.employee
where first_name like '%Dan%'

In [0]:
%sql
-- SELECT DISTINCT department FROM pp_learning.silver.employee;
SELECT department, COUNT(*)
FROM pp_learning.silver.employee
GROUP BY Department;


In [0]:
%sql
SELECT *
FROM pp_learning.silver.employee
ORDER BY salary DESC;

In [0]:
%sql
select count(*) as total_employee
from pp_learning.silver.employee

In [0]:
%sql
select AVG(salary) as av_salary
from pp_learning.silver.employee

In [0]:
%sql
select max(salary)
from pp_learning.silver.employee

In [0]:
%sql
select department,
count(*) as employee_count
from pp_learning.silver.employee
group by department
order by employee_count desc

In [0]:
%sql
select department, avg(salary) as av_salary
from pp_learning.silver.employee
group by department

In [0]:
%sql
select department, count(*) as cnt
from pp_learning.silver.employee
group by department
having count(*) > 10

In [0]:
%sql
select * from pp_learning.silver.employee
where year(joining_date) = 2025

In [0]:
%sql
select first_name, last_name,
datediff(current_date(), joining_date) as exp_days
 from pp_learning.silver.employee


In [0]:
%sql
SELECT first_name,
       last_name,
       salary,
       RANK() OVER(ORDER BY salary DESC) AS rank_no
FROM pp_learning.silver.employee;

In [0]:
%sql
SELECT first_name,
    last_name
       salary,
       ROW_NUMBER() OVER(ORDER BY salary DESC) AS row_num
FROM pp_learning.silver.employee;

In [0]:
%sql
SELECT first_name,
       department,
       salary,
       RANK() OVER(
           PARTITION BY department
           ORDER BY salary DESC
       ) AS dept_rank
FROM pp_learning.silver.employee;

In [0]:
%sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY department
               ORDER BY salary DESC
           ) rn
    FROM pp_learning.silver.employee
)
WHERE rn = 1;

In [0]:
%sql
SELECT e.first_name,
       m.first_name AS manager_name
FROM pp_learning.silver.employee e
LEFT JOIN pp_learning.silver.employee m
ON e.manager_id = m.employee_id;

In [0]:
%sql
SELECT MAX(salary)
FROM pp_learning.silver.employee
WHERE salary <
(
    SELECT MAX(salary)
    FROM pp_learning.silver.employee
);

In [0]:
%sql
SELECT employee_id,
       COUNT(*)
FROM pp_learning.silver.employee
GROUP BY employee_id
HAVING COUNT(*) > 1;

In [0]:
spark.sql("DESCRIBE pp_learning.silver.employee")

In [0]:
%sql
describe pp_learning.silver.employee

In [0]:
%sql
describe history pp_learning.silver.employee

In [0]:
%sql
DESCRIBE DETAIL pp_learning.silver.employee;

In [0]:
%sql
DESCRIBE DETAIL pp_learning.silver.employee;

In [0]:
%sql
OPTIMIZE pp_learning.silver.employee;

In [0]:
%sql
VACUUM pp_learning.silver.employee RETAIN 168 HOURS;

In [0]:
dbutils.fs.ls("s3://cust-e2-unity-catalog/b84f06b4-17d5-4eed-95c9-1bd2476090a2/tables/d62706ab-234e-4b8f-beef-42e4003682ef")

In [0]:
%sql
SELECT employee_id,
       first_name,
       salary,
       SUM(salary)
       OVER
       (
           ORDER BY employee_id
       ) AS running_total
FROM pp_learning.silver.employee;

In [0]:
MERGE INTO pp_learning.silver.employee trg
USING pp_learning.bronze.employee src
ON trg.employee_id = src.employee_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql
INSERT INTO pp_learning.bronze.employee
SELECT *
FROM pp_learning.silver.employee
LIMIT 100000;

In [0]:
%sql
CREATE OR REPLACE TABLE pp_learning.bronze.employee AS
SELECT *
FROM pp_learning.silver.employee
LIMIT 100000;

In [0]:
%sql
SELECT COUNT(*)
FROM pp_learning.bronze.employee;

In [0]:
%sql
select * from pp_learning.bronze.employee limit 1

In [0]:
%sql
select * from pp_learning.silver.employee4 limit 10

In [0]:
%sql
select * from pp_learning.silver.employee 
where employee_id  in (2,3)

In [0]:
%sql
MERGE INTO pp_learning.silver.employee trg
USING pp_learning.silver.employee4 src
ON trg.employee_id = src.employee_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *;